# 03 · Optimización y entrenamiento de ResNet

Objetivo: optimizar los modelos clásicos con Grid Search y entrenar la ResNet 1D con el mejor checkpoint seleccionado por pérdida de validación.

Proyecto de ECG con dos etapas: reposo/actividad y clasificación de cuatro ritmos. Ejecutar los cuadernos **01 → 02 → 03 → 04** con la misma carpeta de resultados. Estos archivos se entregan sin salidas ejecutadas: no contienen métricas inventadas.


## Preparación en Colab
Instalar dependencias, montar Drive y editar las rutas. Subir previamente los datos a Drive; no hace falta subir el ZIP de 2,5 GB si se usa el Parquet Chapman ya consolidado. Para entrenar ResNet, seleccionar un entorno con GPU. El código también admite CPU, con mayor tiempo de ejecución.

Referencias: [entrada y salida de archivos en Colab](https://colab.research.google.com/notebooks/io.ipynb) y [preguntas frecuentes](https://research.google.com/colaboratory/faq.html).


In [ ]:
%pip -q install pyarrow seaborn joblib scipy scikit-learn skl2onnx onnx


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# EDITAR: carpeta del proyecto en tu Google Drive. Usar la misma en los 4 cuadernos.
PROJECT_DIR = Path('/content/drive/MyDrive/TP_IA')
REST_DIR = PROJECT_DIR / 'RestingClass' / 'data_resting'
ACTIVE_DIR = PROJECT_DIR / 'RestingClass' / 'data_not_resting'
CHAPMAN_FILE = PROJECT_DIR / 'Detector' / 'chapman_lead_I_consolidated.parquet'
STAGES = [1, 2]  # Usar [2] si primero quieres trabajar solamente con Chapman.
RESULTS = PROJECT_DIR / 'results_colab'
PREPARED = RESULTS / 'prepared'
MODELS = RESULTS / 'models'
FIGURES = RESULTS / 'figures'
for directory in (RESULTS, PREPARED, MODELS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
assert STAGES and set(STAGES) <= {1, 2}, 'Elegir etapas 1 y/o 2.'
SEED = 42
print('Resultados:', RESULTS)


In [ ]:
import json, random, uuid, hashlib, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, classification_report
from IPython.display import display

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {1: ['Reposo', 'Actividad'], 2: ['Bradicardia sinusal', 'Taquicardia sinusal', 'Fibrilación auricular', 'Ritmo sinusal normal']}
FEATURE_NAMES = {
    1: ['Kurtosis', 'ZCR', 'Crest_Factor', 'Skewness', 'HR', 'RMSSD', 'SDNN', 'EMG_Power', 'Baseline_Power', 'LF_HF_Ratio', 'Morph_Corr', 'Std_Dev'],
    2: ['hr', 'sdnn', 'rmssd', 'pnn50', 'pnn20', 'rr_cv', 'sd1', 'sd2', 'sd1_sd2', 'skew', 'kurt']
}
def save_json(filename, value):
    Path(filename).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def load_prepared(stage):
    filename = PREPARED / f'stage{stage}.npz'
    if not filename.exists():
        raise FileNotFoundError(f'{filename}: ejecutar primero el cuaderno 01 para esta etapa.')
    with np.load(filename, allow_pickle=False) as stored:
        data = {key: stored[key] for key in stored.files}
    return data

def metrics(y, pred, probs):
    # Macro para comparar con igual peso todas las clases, en ambas etapas.
    auc = roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2 else roc_auc_score(y, probs, multi_class='ovr', average='macro')
    return dict(accuracy=accuracy_score(y, pred), precision_macro=precision_score(y, pred, average='macro', zero_division=0), recall_macro=recall_score(y, pred, average='macro', zero_division=0), f1_macro=f1_score(y, pred, average='macro', zero_division=0), auc=auc)

def make_cv(stage, y, groups, folds=3):
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=SEED) if stage == 1 else StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    splits = list(cv.split(np.zeros((len(y), 1)), y, groups if stage == 1 else None))
    for tr, va in splits:
        if len(np.unique(y[tr])) != len(CLASS_NAMES[stage]) or len(np.unique(y[va])) != len(CLASS_NAMES[stage]):
            raise ValueError('Un fold no contiene todas las clases. Revisar sujetos/clases o reducir folds.')
        if stage == 1:
            assert not set(groups[tr]) & set(groups[va]), 'Sujetos compartidos entre folds.'
    return splits

def make_loader(signals, labels, batch_size, shuffle=False):
    # Normalización por señal; no ajusta estadísticas sobre otras muestras.
    a = np.asarray(signals, dtype=np.float32)
    a = (a - a.mean(axis=1, keepdims=True)) / (a.std(axis=1, keepdims=True) + 1e-8)
    dataset = TensorDataset(torch.from_numpy(a).unsqueeze(1), torch.as_tensor(labels, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


class BasicBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 5, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))

class ResNet1D(nn.Module):
    def __init__(self, classes, width, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, width, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(width)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock1D(width, width), BasicBlock1D(width, width))
        self.layer2 = nn.Sequential(BasicBlock1D(width, width*2, 2), BasicBlock1D(width*2, width*2))
        self.layer3 = nn.Sequential(BasicBlock1D(width*2, width*4, 2), BasicBlock1D(width*4, width*4))
        self.layer4 = nn.Sequential(BasicBlock1D(width*4, width*8, 2), BasicBlock1D(width*8, width*8))
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(width*8, classes))
    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.avgpool(x).flatten(1))

def predict_network(model, loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for signals, _ in loader:
            chunks.append(F.softmax(model(signals.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(chunks)

print('Dispositivo:', DEVICE)
versions = {name: __import__(name).__version__ for name in ['numpy', 'pandas', 'sklearn', 'scipy', 'torch', 'joblib']}
versions['python'] = platform.python_version()
save_json(RESULTS / 'environment.json', versions)
display(versions)


## Configuración de ejecución
Grid Search optimiza F1 macro en tres folds; en etapa 1 los folds respetan sujetos. La ResNet recibe las señales crudas normalizadas por ventana y usa hiperparámetros fijos, reducción de learning rate y early stopping: no se afirma haber hecho Grid Search sobre la red. Su entrenamiento puede tardar, especialmente en CPU.


In [ ]:
TRAIN_RESNET = True  # False permite ejecutar solo los modelos clásicos.
EPOCHS = {1: 30, 2: 25}
BATCH_SIZE = {1: 64, 2: 128}
PATIENCE = {1: 6, 2: 5}
EXPORT_ONNX = True
if TRAIN_RESNET and DEVICE.type == 'cpu':
    print('Aviso: ResNet se entrenará en CPU. Para acelerar, seleccionar GPU en Colab.')


## Grid Search con escalado dentro de cada fold
Se guardan todas las combinaciones, medias, desviaciones, ranking y tiempos en CSV. Los hiperparámetros ganadores se registran junto con la versión de datos.


In [ ]:
optimization_rows = []
for stage in STAGES:
    data = load_prepared(stage)
    tr, va = data['train_idx'], data['val_idx']
    cv = make_cv(stage, data['y'][tr], data['groups'][tr])
    rf_params = {'rf__n_estimators': [50, 100], 'rf__max_depth': [3, 4], 'rf__min_samples_leaf': [20, 50], 'rf__max_features': ['sqrt']} if stage == 1 else {'rf__n_estimators': [25, 50], 'rf__max_depth': [4, 5], 'rf__min_samples_leaf': [8, 16], 'rf__max_features': ['sqrt', 0.5]}
    candidates = [
        ('rf', RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=1), rf_params),
        ('lr', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED), {'lr__C': [0.01, 0.1] if stage == 1 else [0.01, 0.1, 1.0], 'lr__solver': ['lbfgs']})
    ]
    for name, classifier, params in candidates:
        search = GridSearchCV(Pipeline([('scaler', StandardScaler()), (name, classifier)]), params, scoring='f1_macro', cv=cv, n_jobs=2, return_train_score=True, error_score='raise')
        search.fit(data['features'][tr], data['y'][tr])
        pd.DataFrame(search.cv_results_).to_csv(RESULTS / f'stage{stage}_{name}_grid_search.csv', index=False)
        joblib.dump(search.best_estimator_, MODELS / f'stage{stage}_{name}.joblib')
        probs = search.best_estimator_.predict_proba(data['features'][va])
        # Umbral fijo 0.5: no se calibra sobre las mismas etiquetas reportadas.
        preds = probs.argmax(axis=1)
        optimization_rows.append(dict(stage=stage, model=name, variant='optimized', **metrics(data['y'][va], preds, probs)))
        save_json(MODELS / f'stage{stage}_{name}_meta.json', dict(dataset_id=str(data['dataset_id']), best_params=search.best_params_, cv_f1_macro=float(search.best_score_), threshold=0.5, classes=CLASS_NAMES[stage]))
        print(f'Etapa {stage} / {name}:', search.best_params_, 'CV F1 macro:', search.best_score_)
        display(pd.DataFrame(search.cv_results_).sort_values('rank_test_score')[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']])
    del data
pd.DataFrame(optimization_rows).to_csv(RESULTS / 'optimized_classic_metrics.csv', index=False)


## ResNet 1D, historial y mejor checkpoint
Se usa la arquitectura residual de los scripts, con ancho inicial 16 y dropout 0,5 en etapa 1; ancho 32 y dropout 0 en etapa 2. Se conserva FocalLoss ponderada con gamma=2 en etapa 1 y cross-entropy en etapa 2. Se guardan artefactos nuevos; los checkpoints originales no se cargan en esta adaptación. La pérdida de validación selecciona el checkpoint; sus métricas posteriores siguen siendo de validación.


In [ ]:
network_rows = []
if TRAIN_RESNET:
    for stage in STAGES:
        data = load_prepared(stage)
        tr, va = data['train_idx'], data['val_idx']
        classes = len(CLASS_NAMES[stage])
        cfg = dict(classes=classes, width=16 if stage == 1 else 32, dropout=0.5 if stage == 1 else 0.0)
        model = ResNet1D(**cfg).to(DEVICE)
        train_loader = make_loader(data['raw'][tr], data['y'][tr], BATCH_SIZE[stage], shuffle=True)
        val_loader = make_loader(data['raw'][va], data['y'][va], BATCH_SIZE[stage])
        weights = None
        if stage == 1:
            counts = np.bincount(data['y'][tr], minlength=classes)
            weights = torch.tensor(len(tr)/(classes*counts), dtype=torch.float32, device=DEVICE)
        criterion = FocalLoss(alpha=weights, gamma=2.0) if stage == 1 else nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3 if stage == 1 else 1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3 if stage == 1 else 2)
        best_loss, stale, history = float('inf'), 0, []
        checkpoint = MODELS / f'stage{stage}_resnet.pth'
        for epoch in range(1, EPOCHS[stage]+1):
            model.train()
            train_loss, train_count, train_pred, train_true = 0.0, 0, [], []
            for signals, labels in train_loader:
                signals, labels = signals.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                logits = model(signals)
                loss = criterion(logits, labels)
                loss.backward(); optimizer.step()
                train_loss += loss.item()*len(labels)
                train_count += len(labels)
                train_pred.extend(logits.detach().argmax(1).cpu().tolist())
                train_true.extend(labels.cpu().tolist())
            model.eval()
            val_loss, val_count, val_pred, val_true = 0.0, 0, [], []
            with torch.no_grad():
                for signals, labels in val_loader:
                    signals, labels = signals.to(DEVICE), labels.to(DEVICE)
                    logits = model(signals)
                    val_loss += criterion(logits, labels).item()*len(labels)
                    val_count += len(labels)
                    val_pred.extend(logits.argmax(1).cpu().tolist())
                    val_true.extend(labels.cpu().tolist())
            row = dict(epoch=epoch, train_loss=train_loss/train_count, val_loss=val_loss/val_count, train_f1_macro=f1_score(train_true, train_pred, average='macro', zero_division=0), val_f1_macro=f1_score(val_true, val_pred, average='macro', zero_division=0), learning_rate=optimizer.param_groups[0]['lr'])
            history.append(row)
            print(f'Etapa {stage}, época {epoch}: loss={row["train_loss"]:.4f}, val_loss={row["val_loss"]:.4f}, val_f1={row["val_f1_macro"]:.4f}')
            pd.DataFrame(history).to_csv(RESULTS / f'stage{stage}_resnet_history.csv', index=False)
            scheduler.step(row['val_loss'])
            if row['val_loss'] < best_loss:
                best_loss, stale = row['val_loss'], 0
                torch.save(model.state_dict(), checkpoint)
                save_json(MODELS / f'stage{stage}_resnet_meta.json', dict(dataset_id=str(data['dataset_id']), config=cfg, best_epoch=epoch, best_val_loss=best_loss, threshold=0.5, classes=CLASS_NAMES[stage]))
            else:
                stale += 1
                if stale >= PATIENCE[stage]:
                    print('Early stopping'); break
        # Corrección: nueva inferencia con los pesos del MEJOR checkpoint.
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
        probs = predict_network(model, val_loader)
        network_rows.append(dict(stage=stage, model='ResNet1D', variant='trained', **metrics(data['y'][va], probs.argmax(1), probs)))
        del data, model, train_loader, val_loader
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    pd.DataFrame(network_rows).to_csv(RESULTS / 'resnet_metrics.csv', index=False)


## Exportación opcional ONNX
Los pipelines clásicos incluyen el escalador; reciben características sin escalar. ResNet recibe señales normalizadas por ventana, con forma `(batch, 1, longitud)`; su salida son logits y hay que aplicar softmax si se necesitan probabilidades. Mantener idéntica extracción de características en Vercel, incluido el cálculo corregido de potencia basal. Los umbrales anteriores de la aplicación no se heredan: aquí se usa argmax (equivale a 0,5 en la etapa binaria).


In [ ]:
if EXPORT_ONNX:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    for stage in STAGES:
        for name in ('rf', 'lr'):
            pipeline = joblib.load(MODELS / f'stage{stage}_{name}.joblib')
            onnx_model = convert_sklearn(pipeline, initial_types=[('float_input', FloatTensorType([None, len(FEATURE_NAMES[stage])]))], target_opset=15, options={id(pipeline.named_steps[name]): {'zipmap': False}})
            (MODELS / f'stage{stage}_{name}.onnx').write_bytes(onnx_model.SerializeToString())
        if TRAIN_RESNET:
            meta = json.loads((MODELS / f'stage{stage}_resnet_meta.json').read_text())
            model = ResNet1D(**meta['config']).to('cpu')
            model.load_state_dict(torch.load(MODELS / f'stage{stage}_resnet.pth', map_location='cpu', weights_only=True))
            model.eval()
            torch.onnx.export(model, torch.randn(1, 1, 4000 if stage == 1 else 5000), str(MODELS / f'stage{stage}_resnet.onnx'), opset_version=15, dynamo=False, input_names=['ecg_input'], output_names=['logits'], dynamic_axes={'ecg_input': {0:'batch_size'}, 'logits':{0:'batch_size'}})
    print('ONNX guardados en', MODELS)


## Qué comparar
Comparar F1 macro de validación del modelo base y del optimizado, y la media/desviación de la búsqueda. Una optimización puede no mejorar la validación. No elegir ni reajustar indefinidamente usando estas mismas etiquetas y presentarlas luego como prueba independiente.
